In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

In [4]:
df=pd.read_csv('data/bank/bank-full.csv',sep=';')
print(df.shape)
print("\nAny missing data")
print(df.isnull().sum())
print('\nClass Distribution')
print(df['y'].value_counts(normalize = True).round(3))

(45211, 17)

Any missing data
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

Class Distribution
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


In [11]:
df_ml = df.copy()

df_ml['y'] = (df_ml['y']=='yes').astype(int)

cat_cols = df_ml.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col])

X = df_ml.drop('y',axis=1)
y = df_ml['y']

X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f"Train: {X_train.shape}; Test: {X_test.shape}")

Train: (36168, 16); Test: (9043, 16)


In [12]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("LOGISTIC REGRESSION")
print(classification_report(y_test, lr_preds, target_names=['No', 'Yes']))

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("RANDOM FOREST")
print(classification_report(y_test, rf_preds, target_names=['No', 'Yes']))

LOGISTIC REGRESSION
              precision    recall  f1-score   support

          No       0.90      0.98      0.94      7985
         Yes       0.58      0.21      0.31      1058

    accuracy                           0.89      9043
   macro avg       0.74      0.60      0.63      9043
weighted avg       0.87      0.89      0.87      9043

RANDOM FOREST
              precision    recall  f1-score   support

          No       0.93      0.97      0.95      7985
         Yes       0.66      0.42      0.51      1058

    accuracy                           0.91      9043
   macro avg       0.79      0.69      0.73      9043
weighted avg       0.89      0.91      0.90      9043



In [13]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)
print("Top 10 Features:")
print(feat_imp.head(10))

Top 10 Features:
duration     0.291153
balance      0.110084
age          0.103281
day          0.090262
month        0.087403
poutcome     0.055356
pdays        0.048788
job          0.048493
campaign     0.038565
education    0.027879
dtype: float64


In [15]:
yes_samples = X_test[rf.predict(X_test)==1].sample(2,random_state=42)
no_samples = X_test[rf.predict(X_test)==0].sample(3,random_state=42)
samples = pd.concat([yes_samples,no_samples])

print("SAMPLE PREDICTIONS")
for i, (idx,row) in enumerate(samples.iterrows()):
    pred = rf.predict([row])[0]
    probab = rf.predict_proba([row])[0][1]
    actual = y_test[idx]
    print(f"Customer {i+1}")
    print(row.to_string())
    print(f"->PREDICTION: {'SUBSCRIBE' if pred==1 else 'NOT SUBSCRIBED'} ")
    print(f"->PROBABILITY: {probab:.2%}")
    print(f"->ACTUAL: {'yes' if actual==1 else 'no'}")


SAMPLE PREDICTIONS
Customer 1
age           41
job            4
marital        2
education      1
default        0
balance      800
housing        0
loan           0
contact        0
day            8
month          1
duration     899
campaign       2
pdays         -1
previous       0
poutcome       3
->PREDICTION: SUBSCRIBE 
->PROBABILITY: 61.00%
->ACTUAL: no
Customer 2
age           43
job            4
marital        2
education      2
default        0
balance      483
housing        0
loan           0
contact        0
day           19
month          1
duration     950
campaign       6
pdays         -1
previous       0
poutcome       3
->PREDICTION: SUBSCRIBE 
->PROBABILITY: 51.00%
->ACTUAL: yes
Customer 3
age            33
job             1
marital         1
education       1
default         0
balance      2279
housing         1
loan            0
contact         1
day            12
month           8
duration      123
campaign        3
pdays          -1
previous        0
poutcome     

In [ ]:
joblib.dump(rf,'model.pk1')
print("Model saved")